# Word2Vec Model for Movie Reviews

This notebook provides a complete summary of the code used to build, train, and deploy a Word2Vec model on a movie review dataset. The final model is presented in an interactive Streamlit application.

## 1. Data Loading and Preprocessing

In [10]:
import pandas as pd
import re
import unicodedata
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK data (only needs to be done once)
nltk.download('punkt')
nltk.download('stopwords')

# Load the dataset
# Make sure 'MovieReview.csv' is in the same directory
df = pd.read_csv('MovieReview.csv')

# We only need the 'review' column for training
df = df.drop('sentiment', axis=1)

print("Dataset shape:", df.shape)
display(df.head())

[nltk_data] Downloading package punkt to /Users/felix/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/felix/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset shape: (25000, 1)


,review
0,With all this stuff going down at the moment w...
1,'The Classic War of the Worlds' by Timothy Hin...
2,The film starts with a manager (Nicholas Bell)...
3,It must be assumed that those who praised this...
4,Superbly trashy and wondrously unpretentious 8...


In [ ]:
import re
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download()
stop_words = stopwords.words('english')

# Converts the unicode file to ascii
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    # creating a space between a word and the punctuation following it
    # eg: "he is a boy." => "he is a boy ."
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)
    # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z?.!]+", " ", w)
    w = re.sub(r'\b\w{0,2}\b', '', w)

    # remove stopword
    mots = word_tokenize(w.strip())
    mots = [mot for mot in mots if mot not in stop_words]
    return ' '.join(mots).strip()

df.review = df.review.apply(lambda x :preprocess_sentence(x))
3df.head()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


KeyboardInterrupt: 

: 

## 2. Tokenization and Data Preparation

In [ ]:
import tensorflow as tf

vocab_size = 10000

# Define and fit the tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(df.review)

# Create word-index mappings
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word

print(f"Vocabulary size: {vocab_size}")

### Create CBOW (Continuous Bag-of-Words) Training Data

In [ ]:
def create_cbow_data(corpus, window_size, vocab_size):
    X, Y = [], []
    for review in corpus:
        # Convert sentences to sequences of integers
        sequences = tokenizer.texts_to_sequences([review])[0]
        if len(sequences) < window_size * 2 + 1:
            continue
        
        # Create context (Y) and target (X) pairs
        for i in range(window_size, len(sequences) - window_size):
            context = sequences[i-window_size:i] + sequences[i+1:i+window_size+1]
            target = sequences[i]
            X.append(target)
            Y.append(context)
            
    return np.array(X), np.array(Y)

WINDOW_SIZE = 5
X, Y = create_cbow_data(df.review, WINDOW_SIZE, vocab_size)

# The target y needs to be reshaped for sparse categorical crossentropy
y = X.reshape(-1, 1)
X_cbow = Y # The context words are the input

print(f"CBOW Input (X) shape: {X_cbow.shape}")
print(f"CBOW Target (y) shape: {y.shape}")

## 3. Model Building and Training

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

embedding_dim = 300

model = Sequential([
    # The model takes integer indices as input, embeds them, and averages the embeddings
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=WINDOW_SIZE*2),
    GlobalAveragePooling1D(),
    # The output layer predicts the center word from the context
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

model.summary()

In [ ]:
# Note: Training is computationally intensive and can take a significant amount of time.
# For a quick demonstration, you might reduce the number of epochs.
history = model.fit(X_cbow, y, batch_size=128, epochs=5, validation_split=0.1)

# Save the trained model
model.save("word2vec.h5")
print("\nModel saved as word2vec.h5")

## 4. Streamlit Application Code (`app.py`)

The following code should be saved in a separate file named `app.py`. It loads the trained model and creates an interactive web interface.

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(layout="wide")

st.title("Word2Vec Model for Semantic Analysis of Movie Reviews")

# --- Caching Functions ---
@st.cache_resource
def load_keras_model():
    """Load the pre-trained Keras model."""
    try:
        model = load_model('word2vec.h5')
        return model
    except Exception as e:
        st.error(f"Error loading H5 model: {e}")
        return None

@st.cache_data
def get_embedding_matrix(_model):
    """Extract the embedding matrix from the model."""
    if _model:
        return _model.layers[0].get_weights()[0]
    return None

# --- Main Application Logic ---
model = load_keras_model()

if model:
    # Load tokenizer and mappings (assuming they were saved during training)
    # For this example, we recreate them. In a real app, save them with pickle.
    df = pd.read_csv('MovieReview.csv')
    vocab_size = 10000
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=vocab_size)
    tokenizer.fit_on_texts(df.review) 
    word2idx = tokenizer.word_index
    idx2word = {i: w for w, i in word2idx.items()}

    embedding_matrix = get_embedding_matrix(model)
    vocab_list = list(word2idx.keys())[:vocab_size]

    st.header("Find Similar Words")
    selected_word = st.selectbox("Select a word to find its closest neighbors:", options=vocab_list)

    if st.button("Find Similar"):
        if selected_word in word2idx:
            word_idx = word2idx[selected_word]
            if word_idx < vocab_size:
                word_vector = embedding_matrix[word_idx]
                similarities = cosine_similarity(word_vector.reshape(1, -1), embedding_matrix)
                # Get top 11 (1 is the word itself), then exclude the first one
                top_indices = np.argsort(similarities[0])[-11:][::-1][1:]
                
                st.subheader(f"Top 10 words similar to '{selected_word}':")
                similar_words = [idx2word.get(i, '[UNK]') for i in top_indices]
                st.write(', '.join(similar_words))
            else:
                st.warning("Word is outside the trained vocabulary size.")
        else:
            st.error("Word not found in vocabulary.")

    st.header("Perform Word Analogies")
    col1, col2, col3 = st.columns(3)
    word1 = col1.text_input("Word 1 (e.g., king)", "king")
    word2 = col2.text_input("Word 2 (to subtract, e.g., man)", "man")
    word3 = col3.text_input("Word 3 (to add, e.g., woman)", "woman")

    if st.button("Calculate Analogy"):
        try:
            vec1 = embedding_matrix[word2idx[word1]]
            vec2 = embedding_matrix[word2idx[word2]]
            vec3 = embedding_matrix[word2idx[word3]]
            
            result_vector = vec1 - vec2 + vec3
            similarities = cosine_similarity(result_vector.reshape(1, -1), embedding_matrix)
            # Exclude the input words themselves from the result
            top_indices = np.argsort(similarities[0])[-10:][::-1]
            result_word = idx2word.get(top_indices[0], '[UNK]')

            st.success(f"Result: **{result_word}**")
        except KeyError as e:
            st.error(f"One of the words ('{e.args[0]}') was not found in the vocabulary.")
else:
    st.warning("Model could not be loaded. Please ensure 'word2vec.h5' is in the correct directory.")



## 5. Deployment Notes

To deploy this application:
1.  **Create a `requirements.txt` file** with the following content:
    ```
    tensorflow
    streamlit
    pandas
    nltk
    scikit-learn
    ```
2.  **Push to GitHub**: Upload `app.py`, `word2vec.h5`, and `requirements.txt` to a GitHub repository. Since `word2vec.h5` is a large file, you must use Git LFS:
    ```bash
    # Initialize Git LFS
    git lfs install
    
    # Track .h5 files
    git lfs track "*.h5"
    
    # Add the .gitattributes file (this is created by the track command)
    git add .gitattributes
    
    # Add, commit, and push your files
    git add app.py word2vec.h5 requirements.txt
    git commit -m "Initial project commit"
    git push origin main
    ```
3.  **Deploy on Streamlit Cloud**: Connect your GitHub repository to Streamlit Cloud and deploy the application.